# 2019 German day-ahead price validation

This notebook compares the simulated EOM clearing prices with the SMARD Germany/Luxembourg hourly day-ahead series. It checks data completeness, handles the daylight-saving-time hour explicitly, reports correlation and price-level errors, and visualizes the results by time, month, and hour of day.

The inputs are expected in `examples/outputs/example_2019_update_base_case_2019/`: `market_meta.csv` and `Gro_handelspreise_201901010000_202001010000_Stunde.xlsx`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.3f}".format

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "examples").is_dir())
results_dir = repo_root / "examples/outputs/example_2019_update_base_case_2019"
market_meta_path = results_dir / "market_meta.csv"
actual_price_path = results_dir / "Gro_handelspreise_201901010000_202001010000_Stunde.xlsx"

assert market_meta_path.exists(), f"Missing simulation results: {market_meta_path}"
assert actual_price_path.exists(), f"Missing SMARD workbook: {actual_price_path}"
print(f"Results folder: {results_dir}")

In [ ]:
# Simulated day-ahead prices: one EOM clearing price for each delivery hour.
market_meta = pd.read_csv(market_meta_path)
simulated = (
    market_meta.loc[market_meta["market_id"].eq("EOM"), ["product_start", "price"]]
    .rename(columns={"product_start": "time", "price": "simulated_eom"})
    .assign(time=lambda frame: pd.to_datetime(frame["time"]))
    .sort_values("time")
    .reset_index(drop=True)
)

# The German/Luxembourg series is the third column in the SMARD export.
actual = pd.read_excel(actual_price_path, sheet_name="Großhandelspreise", header=9, usecols=[0, 2])
actual.columns = ["time", "actual_da"]
actual = (
    actual.assign(
        time=lambda frame: pd.to_datetime(frame["time"], dayfirst=True, errors="coerce"),
        actual_da=lambda frame: pd.to_numeric(frame["actual_da"], errors="coerce"),
    )
    .dropna(subset=["time", "actual_da"])
    .reset_index(drop=True)
)

# SMARD retains both autumn daylight-saving 02:00 hours and omits spring 02:00.
# The simulator uses a naive hourly index, so we average the duplicated autumn
# observation and compare only common local-clock timestamps.
actual_by_clock = actual.groupby("time", as_index=False)["actual_da"].mean()
paired = simulated.merge(actual_by_clock, on="time", how="inner", validate="one_to_one")
paired["error"] = paired["simulated_eom"] - paired["actual_da"]
paired["month"] = paired["time"].dt.to_period("M")
paired["hour"] = paired["time"].dt.hour

print(f"Simulated EOM hours: {len(simulated):,}")
print(f"SMARD price rows: {len(actual):,}; unique local-clock hours: {len(actual_by_clock):,}")
print(f"Matched hours: {len(paired):,} ({paired.time.min():%Y-%m-%d %H:%M} to {paired.time.max():%Y-%m-%d %H:%M})")

In [ ]:
# Reproducible robustness checks. Change MIN_PEARSON if a different
# acceptance target is agreed for future calibrated scenarios.
MIN_MATCHED_HOURS = 8_700
MIN_PEARSON = 0.65

metrics = pd.Series({
    "matched hours": len(paired),
    "Pearson correlation": paired["simulated_eom"].corr(paired["actual_da"], method="pearson"),
    "Spearman correlation": paired["simulated_eom"].corr(paired["actual_da"], method="spearman"),
    "mean simulated price (EUR/MWh)": paired["simulated_eom"].mean(),
    "mean actual price (EUR/MWh)": paired["actual_da"].mean(),
    "mean bias, simulated - actual (EUR/MWh)": paired["error"].mean(),
    "MAE (EUR/MWh)": paired["error"].abs().mean(),
    "RMSE (EUR/MWh)": np.sqrt((paired["error"] ** 2).mean()),
})
display(metrics.to_frame("value"))

checks = pd.DataFrame({
    "check": ["All EOM timestamps unique", "Sufficient common hours", "Finite paired prices", "Pearson target"],
    "passed": [
        simulated["time"].is_unique,
        len(paired) >= MIN_MATCHED_HOURS,
        np.isfinite(paired[["simulated_eom", "actual_da"]]).all().all(),
        metrics["Pearson correlation"] >= MIN_PEARSON,
    ],
})
display(checks)
assert checks["passed"].all(), "At least one robustness check failed."

In [ ]:
# Hourly comparison for a representative winter week and price scatter.
week = paired.loc[paired["time"].between("2019-01-14", "2019-01-21")].copy()
fig, axes = plt.subplots(2, 1, figsize=(14, 10), constrained_layout=True)

axes[0].plot(week["time"], week["actual_da"], label="Actual Germany/Luxembourg DA", lw=1.8)
axes[0].plot(week["time"], week["simulated_eom"], label="Simulated EOM", lw=1.4)
axes[0].set(title="Hourly day-ahead prices: representative winter week", ylabel="EUR/MWh")
axes[0].legend()

axes[1].scatter(paired["actual_da"], paired["simulated_eom"], s=7, alpha=0.22)
lower = min(paired[["actual_da", "simulated_eom"]].min())
upper = max(paired[["actual_da", "simulated_eom"]].max())
axes[1].plot([lower, upper], [lower, upper], "k--", lw=1, label="Perfect agreement")
axes[1].set(
    title=f"All matched hours: Pearson r = {metrics['Pearson correlation']:.3f}",
    xlabel="Actual day-ahead price (EUR/MWh)",
    ylabel="Simulated EOM price (EUR/MWh)",
)
axes[1].legend()
plt.show()

In [ ]:
# Monthly robustness: does the relationship remain stable throughout the year?
monthly = paired.groupby("month").agg(
    pearson=("simulated_eom", lambda series: series.corr(paired.loc[series.index, "actual_da"])),
    simulated_mean=("simulated_eom", "mean"),
    actual_mean=("actual_da", "mean"),
    bias=("error", "mean"),
    mae=("error", lambda series: series.abs().mean()),
)
display(monthly)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
monthly[["simulated_mean", "actual_mean"]].plot(kind="bar", ax=axes[0])
axes[0].set(title="Monthly mean price", xlabel="Month", ylabel="EUR/MWh")
axes[0].legend(["Simulated EOM", "Actual DA"])
monthly[["pearson", "mae"]].plot(kind="bar", secondary_y="mae", ax=axes[1])
axes[1].set(title="Monthly correlation and absolute error", xlabel="Month", ylabel="Pearson correlation")
axes[1].right_ax.set_ylabel("MAE (EUR/MWh)")
plt.show()

In [ ]:
# Average day-ahead delivery-hour shape: this is not an intraday-market price.
hourly = paired.groupby("hour")[["simulated_eom", "actual_da"]].mean()
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hourly.index, hourly["actual_da"], marker="o", label="Actual DA")
ax.plot(hourly.index, hourly["simulated_eom"], marker="o", label="Simulated EOM")
ax.set(
    title="Average 2019 day-ahead price by delivery hour",
    xlabel="Delivery hour",
    ylabel="EUR/MWh",
    xticks=range(24),
)
ax.legend()
plt.show()

print(
    f"Interpretation: r={metrics['Pearson correlation']:.3f} indicates a positive hourly co-movement; "
    f"the mean bias of {metrics['mean bias, simulated - actual (EUR/MWh)']:.2f} EUR/MWh "
    "shows whether the model price level is systematically too high or too low."
)

## CRM positive and negative reserve outcomes

The annual case clears both CRM markets in four-hour capacity blocks. The following cells validate market coverage and volume balance, then compare clearing-price and procured-capacity outcomes. No external CRM benchmark is used here; this section assesses the internal consistency and behaviour of the simulated markets.

## CRM capacity-price analysis

`CRM_pos` and `CRM_neg` identify upward and downward balancing directions. Both are capacity-remuneration markets, so their clearing prices must be non-negative. This is separate from balancing-energy prices, which can be signed.

The model’s downward-CRM bid calculation was corrected after the initial annual result: the old result file contains negative `CRM_neg` capacity prices and is deliberately rejected by the check below. Regenerate `market_meta.csv` with the corrected model before executing this section.

In [ ]:
crm_markets = ["CRM_pos", "CRM_neg"]
crm = market_meta.loc[market_meta["market_id"].isin(crm_markets)].copy()
crm["time"] = pd.to_datetime(crm["product_start"])
crm["month"] = crm["time"].dt.to_period("M")
crm["duration_hours"] = (pd.to_datetime(crm["product_end"]) - crm["time"]).dt.total_seconds() / 3600
crm["capacity_balance"] = crm["supply_volume"] - crm["demand_volume"]

assert set(crm_markets).issubset(set(crm["market_id"])), "One or both CRM markets are missing."
assert (crm["price"] >= -1e-9).all(), (
    "CRM capacity remuneration cannot be negative. This market_meta.csv was "
    "generated before the CRM_neg sign correction; regenerate the simulation."
)
assert crm.groupby("market_id")["time"].apply(lambda series: series.is_unique).all(), "Duplicate CRM delivery blocks found."
assert np.isclose(crm["capacity_balance"], 0).all(), "CRM demand was not balanced by cleared supply."
assert crm["duration_hours"].eq(4).all(), "Unexpected CRM product duration."

crm_summary = crm.groupby("market_id").agg(
    cleared_blocks=("time", "size"),
    first_delivery=("time", "min"),
    last_delivery=("time", "max"),
    mean_price=("price", "mean"),
    median_price=("price", "median"),
    min_price=("price", "min"),
    max_price=("price", "max"),
    mean_cleared_capacity_mw=("supply_volume", "mean"),
    total_capacity_energy_mwh=("supply_volume_energy", "sum"),
    max_abs_capacity_imbalance_mw=("capacity_balance", lambda series: series.abs().max()),
)
display(crm_summary)

In [ ]:
# Price dynamics and cleared capacity. A 42-block window corresponds to one week.
weekly = crm.set_index("time").groupby("market_id")[["price", "supply_volume"]].rolling(42, min_periods=1).mean().reset_index()
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
for market, frame in weekly.groupby("market_id"):
    label = "CRM positive" if market == "CRM_pos" else "CRM negative"
    axes[0].plot(frame["time"], frame["price"], label=label)
    axes[1].plot(frame["time"], frame["supply_volume"], label=label)
axes[0].set(title="CRM weekly rolling clearing-price mean", ylabel="EUR/MW")
axes[1].set(title="CRM weekly rolling cleared capacity", ylabel="MW", xlabel="Delivery date")
axes[0].legend()
axes[1].legend()
plt.show()

In [ ]:
# Monthly outcomes and the price distributions reveal scarcity and negative-price blocks.
crm_monthly = crm.groupby(["month", "market_id"]).agg(
    mean_price=("price", "mean"),
    mean_cleared_capacity_mw=("supply_volume", "mean"),
    price_volatility=("price", "std"),
).reset_index()
display(crm_monthly.pivot(index="month", columns="market_id", values=["mean_price", "mean_cleared_capacity_mw"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for market, frame in crm_monthly.groupby("market_id"):
    label = "CRM positive" if market == "CRM_pos" else "CRM negative"
    axes[0].plot(frame["month"].astype(str), frame["mean_price"], marker="o", label=label)
for market, frame in crm.groupby("market_id"):
    label = "CRM positive" if market == "CRM_pos" else "CRM negative"
    axes[1].hist(frame["price"], bins=45, alpha=0.55, label=label)
axes[0].set(title="Monthly mean CRM clearing price", xlabel="Month", ylabel="EUR/MW")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()
axes[1].set(title="CRM clearing-price distribution", xlabel="EUR/MW", ylabel="Number of four-hour blocks")
axes[1].legend()
plt.show()

print("Both CRM markets clear every four hours and have zero supply-demand imbalance in this result. "
      "Positive and negative reserve prices must be interpreted separately because they procure opposite flexibility directions.")